In [1]:
# 01_data_preparation.ipynb

# -----------------------------
# 1. Imports
# -----------------------------
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from pathlib import Path
import pickle

# -----------------------------
# 2. Load raw data
# -----------------------------
# Adjust paths to your actual file locations
events = pd.read_csv("events.csv")
ip1 = pd.read_csv("item_properties_part1.csv")
ip2 = pd.read_csv("item_properties_part2.csv")  # if exists
item_props = pd.concat([ip1, ip2], ignore_index=True)
category_tree = pd.read_csv("category_tree.csv")

print("Events shape:", events.shape)
print("Item properties shape:", item_props.shape)
print("Category tree shape:", category_tree.shape)

events.head()


Events shape: (2756101, 5)
Item properties shape: (20275902, 4)
Category tree shape: (1669, 2)


,timestamp,visitorid,event,itemid,transactionid
0,1433221332117,257597,view,355908,NaN
1,1433224214164,992329,view,248676,NaN
2,1433221999827,111016,view,318965,NaN
3,1433221955914,483717,view,253185,NaN
4,1433221337106,951259,view,367447,NaN


In [2]:
# ------------------------------------------------------------
# 2. CLEAN EVENTS + APPLY IMPLICIT WEIGHTS
# ------------------------------------------------------------

# Keep only valid events
valid_events = ["view", "addtocart", "transaction"]
events = events[events["event"].isin(valid_events)].copy()

event_weights = {
    "view": 1.0,
    "addtocart": 3.0,
    "transaction": 5.0
}

events["weight"] = events["event"].map(event_weights)

print("Events after filtering:", events.shape)


Events after filtering: (2756101, 6)


In [3]:
# ------------------------------------------------------------
# 3. BUILD USER AND ITEM INDICES (CRITICAL STEP)
# ------------------------------------------------------------

# Build item index ONLY from events (MUST BE 235,061)
item_ids = events["itemid"].unique()
item_id_to_idx = {i: j for j, i in enumerate(item_ids)}
idx_to_item_id = {j: i for i, j in item_id_to_idx.items()}
n_items = len(item_ids)

# Build user index
user_ids = events["visitorid"].unique()
user_id_to_idx = {u: i for i, u in enumerate(user_ids)}
idx_to_user_id = {i: u for u, i in user_id_to_idx.items()}
n_users = len(user_ids)

print("Users:", n_users)
print("Items:", n_items)   # MUST be 235,061


Users: 1407580
Items: 235061


In [4]:
# ------------------------------------------------------------
# 4. MAP EVENTS TO INDICES
# ------------------------------------------------------------
events["user_idx"] = events["visitorid"].map(user_id_to_idx)
events["item_idx"] = events["itemid"].map(item_id_to_idx)

events.head()


,timestamp,visitorid,event,itemid,transactionid,weight,user_idx,item_idx
0,1433221332117,257597,view,355908,NaN,1.0,0,0
1,1433224214164,992329,view,248676,NaN,1.0,1,1
2,1433221999827,111016,view,318965,NaN,1.0,2,2
3,1433221955914,483717,view,253185,NaN,1.0,3,3
4,1433221337106,951259,view,367447,NaN,1.0,4,4


In [5]:
# ------------------------------------------------------------
# 5. BUILD USER–ITEM INTERACTION MATRIX
# ------------------------------------------------------------

interaction_df = (
    events.groupby(["user_idx", "item_idx"])["weight"]
    .sum()
    .reset_index()
)

interactions = csr_matrix(
    (interaction_df["weight"], (interaction_df["user_idx"], interaction_df["item_idx"])),
    shape=(n_users, n_items)
)

print("Interaction matrix:", interactions.shape)
print("Non-zero entries:", interactions.nnz)


Interaction matrix: (1407580, 235061)
Non-zero entries: 2145179


In [6]:
# ------------------------------------------------------------
# 6. FILTER ITEM_PROPERTIES TO ONLY ITEMS IN EVENTS
# ------------------------------------------------------------

valid_item_ids = set(item_id_to_idx.keys())

item_props_filtered = item_props[item_props["itemid"].isin(valid_item_ids)].copy()
print("Filtered item_properties:", item_props_filtered.shape)

# Sort so last snapshot is latest
item_props_filtered = item_props_filtered.sort_values(
    ["itemid", "property", "timestamp"]
)

# Keep latest snapshot per (itemid, property)
item_props_latest = (
    item_props_filtered.groupby(["itemid", "property"]).tail(1)
)

print("Latest snapshots:", item_props_latest.shape)


Filtered item_properties: (10180153, 4)
Latest snapshots: (5369446, 4)


In [7]:
# ------------------------------------------------------------
# 7. BUILD TEXT FEATURES FOR ITEMS
# ------------------------------------------------------------

item_text_df = (
    item_props_latest
    .groupby("itemid")["value"]
    .apply(lambda vals: " ".join(map(str, vals)))
    .reset_index()
    .rename(columns={"value": "raw_text"})
)

print("Item text rows:", item_text_df.shape)


Item text rows: (185246, 2)


In [8]:
# ------------------------------------------------------------
# 8. ALIGN TEXT FEATURES WITH item_idx (CRITICAL)
# ------------------------------------------------------------

item_text_df["item_idx"] = item_text_df["itemid"].map(item_id_to_idx)
item_text_df = item_text_df.dropna(subset=["item_idx"])
item_text_df["item_idx"] = item_text_df["item_idx"].astype(int)

# Build final aligned items table
items = pd.DataFrame({
    "item_idx": np.arange(n_items),
    "itemid": [idx_to_item_id[i] for i in range(n_items)]
}).merge(item_text_df, on="item_idx", how="left")

items["raw_text"] = items["raw_text"].fillna("")

print("Final items table:", items.shape)   # MUST be (235061, ...)
items.head()


Final items table: (235061, 4)


,item_idx,itemid_x,itemid_y,raw_text
0,0,355908,355908.0,726612 n1020.000 424566 679677 519769 264157 2...
1,1,248676,248676.0,961511 679677 519769 769062 857982 540717 1407...
2,2,318965,NaN,
3,3,253185,253185.0,769062 469355 679677 769062 519769 769062 n120...
4,4,367447,367447.0,119932 754228 801383 471403 801383 693640 9860...


In [9]:
# ------------------------------------------------------------
# 9. SAVE ARTIFACTS FOR NEXT NOTEBOOKS
# ------------------------------------------------------------

from scipy import sparse
Path("artifacts").mkdir(exist_ok=True)

sparse.save_npz("input/artifacts/interactions.npz", interactions)
items.to_csv("input/artifacts/items_aligned.csv", index=False)

with open("input/artifacts/user_id_to_idx.pkl", "wb") as f:
    pickle.dump(user_id_to_idx, f)

with open("input/artifacts/item_id_to_idx.pkl", "wb") as f:
    pickle.dump(item_id_to_idx, f)

print("Artifacts saved successfully.")


Artifacts saved successfully.
